# 04 deployment interface

**Owner:** Thomas. Original notebook cells: 99.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# THOMAS: Complete deployment-facing AI chat interface

import json

# Verify upstream components
required_components = [
    "eye_style_chat",
    "eye_style_capabilities",
    "create_mvp_workflow_agent",
    "mvp_gateway_summary",
    "MVP_DATASET_VERSION",
    "MVP_REFERENCE_DATE",
    "normalize_website_role",
    "validate_subject",
    "validate_website_answer",
    "json_compatible",
    "ROLE_WORKFLOWS",
    "WORKFLOW_OPERATIONS",
    "MAX_QUESTION_LENGTH",
    "RISK_SCORE_NOTICE",
    "AgentError"
]

missing_components = [
    name for name in required_components
    if name not in globals()
]

if missing_components:

    raise RuntimeError(
        "Missing required components: "
        + ", ".join(missing_components)
    )

if not mvp_gateway_summary.get("all_tests_passed"):

    raise RuntimeError(
        "Operational gateway validation has not passed."
    )


# Deployment-facing AI chat
def eye_style_deployment_chat(
    question,
    workflow,
    authenticated_role,
    authenticated_subject
):

    # Identity and role must eventually originate
    # from a verified Firebase authentication token.

    if not isinstance(question, str):

        raise AgentError("Question must be text.")

    question = question.strip()

    if not question:

        raise AgentError("Question cannot be empty.")

    if len(question) > MAX_QUESTION_LENGTH:

        raise AgentError(
            "Question exceeds length limit."
        )

    role = normalize_website_role(
        authenticated_role
    )

    subject = validate_subject(
        authenticated_subject
    )

    if not isinstance(workflow, str):

        raise AgentError("Invalid workflow.")

    workflow = workflow.strip()

    # RAG uses the already tested Cell 93 interface.
    # No additional embedding model is initialized.

    if workflow == "RAG":

        response = eye_style_chat(
            question=question,
            workflow="RAG",
            authenticated_role=role,
            authenticated_subject=subject
        )

        response["data_source"] = (
            "synthetic_knowledge_corpus"
        )

        return json_compatible(response)

    # Preserve operational role restrictions
    if workflow not in ROLE_WORKFLOWS.get(
        role, set()
    ):

        raise AgentError(
            "Access to this workflow is denied."
        )

    # Use Daniel's validated operational dataset
    specialist, gateway = create_mvp_workflow_agent(
        staff_role=role,
        workflow=workflow,
        subject=subject
    )

    agent_question = (
        question
        + "\nSynthetic dataset reference date: "
        + MVP_REFERENCE_DATE
        + ". Use authorized synthetic records only. "
        + "Do not modify records or contact patients. "
        + "Describe no-show scores as model risk scores, "
        + "not calibrated probabilities. "
        + "If requested information is unavailable, "
        + "explicitly state that it is unavailable. "
        + "Do not invent email, telephone, revenue, "
        + "sales, or other unsupported dashboard metrics. "
        + "After retrieving sufficient information, "
        + "return a final written answer."
    )

    result = specialist.ask(agent_question)

    if not isinstance(result, dict):

        raise AgentError("Invalid agent response.")

    answer = validate_website_answer(
        result.get("answer")
    )

    sources = list(
        dict.fromkeys(gateway.read_trace)
    )

    # Verify that only authorized operations were used
    if not set(sources).issubset(
        WORKFLOW_OPERATIONS[workflow]
    ):

        raise AgentError(
            "Unauthorized operation detected."
        )

    notices = [
        "Synthetic demonstration data only."
    ]

    if "GetNoShowRisk" in sources:

        notices.append(RISK_SCORE_NOTICE)

    response = {
        "success": True,
        "answer": answer,
        "workflow": workflow,
        "role": role,
        "reports": result.get("reports", []),
        "sources": sources,
        "notices": notices,
        "mode": "demo",
        "data_source": MVP_DATASET_VERSION,
        "reference_date": MVP_REFERENCE_DATE,
        "requires_staff_review": True
    }

    return json_compatible(response)


# Deployment capability information
def eye_style_deployment_capabilities(
    authenticated_role,
    authenticated_subject
):

    response = eye_style_capabilities(
        authenticated_role,
        authenticated_subject
    )

    response.update({
        "operational_dataset": MVP_DATASET_VERSION,
        "reference_date": MVP_REFERENCE_DATE,
        "saved_no_show_scores_available": True,
        "email_tracking_available": False,
        "phone_tracking_available": False,
        "text_tracking_available": False,
        "full_financial_analytics_available": False,
        "automatic_patient_contact": False,
        "replace_existing_website_metrics": False
    })

    return json_compatible(response)


print("=" * 60)
print("EYE STYLE AI DEPLOYMENT INTERFACE")
print("=" * 60)

print("Scheduling Agent: Configured")
print("Insurance Agent: Configured")
print("Optical Agent: Configured")
print("RAG Assistant: Configured")
print("Operational dataset:", MVP_DATASET_VERSION)
print("Dataset reference date:", MVP_REFERENCE_DATE)
print("Saved no-show scores: Enabled")
print("Role-based permissions: Preserved")
print("Read-only restrictions: Preserved")
print("Existing website metrics: Unchanged")
print("Live Crystal connection: Disabled")
print("Firebase authentication: Pending")
print("HTTP endpoint deployment: Pending")

print("\nDeployment-facing functions initialized.")